# Детекция ботов по событиям внутри суточного окна

Задача: для каждой `cookie_id` из `data/test.csv` предсказать `score` от 0 до 1 — вероятность того, что кука принадлежит сервису автоматизированного сбора данных. Метрика — Precision при Recall ≥ 70% (`metric.py`).

Структура ноутбука:
1. Настройки и загрузка данных
2. Очистка данных
3. Исследование данных (EDA) и проверки на утечки
4. Окно наблюдения
5. Сравнение ботов и людей
6. Признаки
7. Валидация
8. Журнал экспериментов
9. Финальная модель и `submission.csv`

## 1. Настройки и загрузка данных

In [1]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier

from metric import precision_at_recall  # официальная реализация метрики

SEED = 0
SEEDS = list(range(10))     # каждый вариант обучается на 10 seed, итоговый score — среднее по ним
VALID_START = '2026-04-17'  # временной holdout: окна начиная с этой даты
EVENTS_PATH = 'data/events.csv.gz'

In [2]:
date_cols = ['cookie_created_at', 'window_start_ts', 'window_end_ts']
train = pd.read_csv('data/train.csv', parse_dates=date_cols)
test = pd.read_csv('data/test.csv', parse_dates=date_cols)
events = pd.read_csv(EVENTS_PATH, parse_dates=['event_ts'])

print(train.shape, test.shape, events.shape)
print('доля ботов в train:', train.target.mean().round(4))
print('окна train:', train.window_start_ts.min().date(), '—', train.window_start_ts.max().date())
print('окна test: ', test.window_start_ts.min().date(), '—', test.window_start_ts.max().date())

(11091, 5) (4909, 4) (328905, 14)
доля ботов в train: 0.0811
окна train: 2026-04-06 — 2026-04-19
окна test:  2026-04-20 — 2026-04-26


Каждая кука наблюдается ровно одни календарные сутки. Test идёт сразу после train по времени (7 дней после 14 дней train), пересечений `cookie_id` между ними нет.

## 2. Очистка данных

- `platform` записана в разном регистре (`android` / `Android` / `ANDROID`) и с синонимами (`iphone` = `ios`, `desktop` = `web` — одинаковые User-Agent). Приводим к трём значениям. Исходное написание сохраняем в `platform_raw` только для проверки в EDA.
- Строки файла событий перемешаны, поэтому сортируем по куке и времени: без этого паузы между событиями считаются неверно.
- Полные дубликаты событий (~1,5%) удаляем: это шум логирования (см. EDA ниже).

In [3]:
events['platform_raw'] = events.platform
events['platform'] = events.platform.str.lower().replace({'iphone': 'ios', 'desktop': 'web'})
events = events.sort_values(['cookie_id', 'event_ts']).reset_index(drop=True)

dup_cols = [c for c in events.columns if c != 'platform_raw']
is_dup = events.duplicated(subset=dup_cols)
print('полных дубликатов:', is_dup.sum(), f'({is_dup.mean():.1%})')
events = events[~is_dup].reset_index(drop=True)

events.platform.value_counts()

полных дубликатов: 4868 (1.5%)


platform
web        182943
android    124984
ios         16110
Name: count, dtype: int64

## 3. Исследование данных (EDA)

### 3.1. Типы событий и пропуски

In [4]:
print(events.event_name.value_counts(), '\n')
print(events.groupby('eid').event_name.unique(), '\n')
print('пропуски по колонкам:')
print(events.isna().mean().round(3))

event_name
item_view               118989
search_results_view      98928
photo_swipe              35962
favorite_add             18739
seller_page_view         17180
contact_phone_show       11154
captcha_shown             7829
login                     6177
contact_chat_open         6047
contact_message_sent      3032
Name: count, dtype: int64 

eid
100     [search_results_view]
200               [item_view]
210             [photo_swipe]
220        [seller_page_view]
300      [contact_phone_show]
301       [contact_chat_open]
303    [contact_message_sent]
400            [favorite_add]
500                   [login]
900           [captcha_shown]
Name: event_name, dtype: object 

пропуски по колонкам:
cookie_id        0.000
event_ts         0.000
eid              0.000
event_name       0.000
platform         0.000
user_agent       0.000
item_id          0.349
item_category    0.100
item_location    0.072
seller_type      0.407
search_query     0.695
search_page      0.695
pointer_x      

Пропуски объясняются типами событий: `search_query` и `search_page` заполнены только у поиска (~30% событий), `pointer_x/y` отсутствуют у мобильных приложений.

### 3.2. Где лежат события относительно окна: проверка на утечку

В файле событий есть не только события внутри окна. Смотрим, сколько событий каждого типа лежит до окна, внутри и после.

In [5]:
windows = pd.concat([
    train[['cookie_id', 'window_start_ts', 'window_end_ts']],
    test[['cookie_id', 'window_start_ts', 'window_end_ts']],
])
pos = events.merge(windows, on='cookie_id')
pos['position'] = np.select(
    [pos.event_ts < pos.window_start_ts, pos.event_ts >= pos.window_end_ts],
    ['до окна', 'после окна'],
    default='внутри окна',
)
pd.crosstab(pos.event_name, pos.position)

position,внутри окна,после окна
event_name,,
captcha_shown,0,7829
contact_chat_open,5479,568
contact_message_sent,2778,254
contact_phone_show,10145,1009
favorite_add,17012,1727
item_view,106447,12542
login,5578,599
photo_swipe,32570,3392
search_results_view,88478,10450


In [6]:
# капча после окна почти идеально отделяет ботов — именно поэтому её нельзя использовать
cap_cookies = pos.loc[pos.event_name == 'captcha_shown', 'cookie_id'].unique()
train.assign(got_captcha=train.cookie_id.isin(cap_cookies)).groupby('target').got_captcha.mean()

target
0    0.012853
1    0.490545
Name: got_captcha, dtype: float64

**Выводы:**
- Событий до окна нет.
- ~12% событий лежат **после** окна, причём все `captcha_shown` — только после окна. Капчу после окна получили ~49% ботов и ~1% людей: это реакция антибот-системы, то есть информация из будущего. Использовать её нельзя.
- Остальные события после окна тоже из будущего. Поэтому все признаки считаются строго по событиям внутри окна, капча в признаки не попадает.

### 3.3. Время создания куки: проверка на утечку

In [7]:
for name, df in [('train', train), ('test', test)]:
    print(name, 'кук, созданных не раньше конца окна:', (df.cookie_created_at >= df.window_end_ts).sum())

train кук, созданных не раньше конца окна: 0
test кук, созданных не раньше конца окна: 0


**Вывод:** у всех кук `cookie_created_at` раньше конца окна, поэтому возраст куки можно использовать как признак.

### 3.4. Написание платформы

In [8]:
chk = events.merge(train[['cookie_id', 'target']], on='cookie_id')
chk = chk.groupby('cookie_id').agg(target=('target', 'first'), n_spellings=('platform_raw', 'nunique'))
chk.groupby('target').n_spellings.agg(mean='mean', share_multi=lambda s: (s > 1).mean())

,mean,share_multi
target,,
0,3.310341,0.976060
1,3.651835,0.997775


**Вывод:** разнобой в написании `platform` есть у 96% людей и 99% ботов, то есть это шум логирования, а не сигнал. Значения нормализованы, признак из написаний не строится.

## 4. Окно наблюдения

Оставляем для каждой куки только события внутри её окна: `window_start_ts <= event_ts < window_end_ts`.

In [9]:
def events_in_window(events, meta):
    """Оставляет события каждой куки внутри её окна: window_start_ts <= event_ts < window_end_ts."""
    ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    ev = ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]
    return ev.sort_values(['cookie_id', 'event_ts'], kind='mergesort').reset_index(drop=True)


ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)
print(len(ev_tr), len(ev_te))
print('кук без событий в окне:', (~train.cookie_id.isin(ev_tr.cookie_id)).sum(), (~test.cookie_id.isin(ev_te.cookie_id)).sum())

195482 88348
кук без событий в окне: 0 0


### 4.1. Дубликаты событий

Проверяем, не являются ли дубли признаком бота (сравниваем на исходных событиях окна до удаления дублей).

In [10]:
raw = pd.read_csv(EVENTS_PATH, parse_dates=['event_ts'])
raw = events_in_window(raw, train).merge(train[['cookie_id', 'target']], on='cookie_id')
raw['is_dup'] = raw.drop(columns=['target']).duplicated()
d = raw.groupby('cookie_id').agg(target=('target', 'first'), dup_share=('is_dup', 'mean'), any_dup=('is_dup', 'max'))
d.groupby('target')[['dup_share', 'any_dup']].mean().round(4)

,dup_share,any_dup
target,,
0,0.0134,0.2033
1,0.0144,0.3204


**Вывод:** доля дублей среди событий куки у ботов и людей одинаковая. То, что хотя бы один дубль чаще встречается у ботов, объясняется большим числом событий у ботов. Это шум логирования: дубли удалены, признак из них не строится.

## 5. Сравнение ботов и людей

In [11]:
def show_cookie(cookie_id):
    """Показывает события одной куки внутри окна по порядку времени."""
    return ev_tr[ev_tr.cookie_id == cookie_id]


show_cookie(train[train.target == 1].cookie_id.iloc[0]).head(15)

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y,platform_raw,window_start_ts,window_end_ts
171742,ck_e00e1e6c2a2ab4ac,2026-04-06 21:09:49,100,search_results_view,android,Avito/118.0 (Android 11; SM-A515F) okhttp/4.11.0,NaN,uslugi,sankt-peterburg,NaN,сборка мебели,3.0,NaN,NaN,android,2026-04-06,2026-04-07
171743,ck_e00e1e6c2a2ab4ac,2026-04-06 21:39:36,100,search_results_view,android,Avito/118.0 (Android 11; SM-A515F) okhttp/4.11.0,NaN,uslugi,barnaul,NaN,мастер на все руки,5.0,NaN,NaN,android,2026-04-06,2026-04-07
171744,ck_e00e1e6c2a2ab4ac,2026-04-06 21:39:38,100,search_results_view,android,Avito/118.0 (Android 11; SM-A515F) okhttp/4.11.0,NaN,zapchasti,sankt-peterburg,NaN,шины зимние,4.0,NaN,NaN,ANDROID,2026-04-06,2026-04-07
171745,ck_e00e1e6c2a2ab4ac,2026-04-06 21:39:45,300,contact_phone_show,android,Avito/118.0 (Android 11; SM-A515F) okhttp/4.11.0,1024813.0,avtomobili,sankt-peterburg,pro,NaN,NaN,NaN,NaN,Android,2026-04-06,2026-04-07
171746,ck_e00e1e6c2a2ab4ac,2026-04-06 21:40:03,210,photo_swipe,android,Avito/118.0 (Android 11; SM-A515F) okhttp/4.11.0,1021391.0,kvartiry_prodazha,sankt-peterburg,private,NaN,NaN,NaN,NaN,android,2026-04-06,2026-04-07
171747,ck_e00e1e6c2a2ab4ac,2026-04-06 21:40:17,100,search_results_view,android,Avito/118.0 (Android 11; SM-A515F) okhttp/4.11.0,NaN,NaN,sankt-peterburg,NaN,джинсы levis,4.0,NaN,NaN,ANDROID,2026-04-06,2026-04-07
171748,ck_e00e1e6c2a2ab4ac,2026-04-06 21:40:33,400,favorite_add,android,Avito/118.0 (Android 11; SM-A515F) okhttp/4.11.0,1016533.0,zapchasti,sankt-peterburg,pro,NaN,NaN,NaN,NaN,android,2026-04-06,2026-04-07
171749,ck_e00e1e6c2a2ab4ac,2026-04-06 21:40:40,200,item_view,android,Avito/118.0 (Android 11; SM-A515F) okhttp/4.11.0,1032690.0,uslugi,sankt-peterburg,pro,NaN,NaN,NaN,NaN,android,2026-04-06,2026-04-07
171750,ck_e00e1e6c2a2ab4ac,2026-04-06 23:04:32,200,item_view,android,Avito/118.0 (Android 11; SM-A515F) okhttp/4.11.0,1055889.0,zapchasti,sankt-peterburg,private,NaN,NaN,NaN,NaN,Android,2026-04-06,2026-04-07
171751,ck_e00e1e6c2a2ab4ac,2026-04-06 23:04:40,100,search_results_view,android,Avito/118.0 (Android 11; SM-A515F) okhttp/4.11.0,NaN,zapchasti,sankt-peterburg,NaN,шины зимние,2.0,NaN,NaN,Android,2026-04-06,2026-04-07


In [12]:
ev = ev_tr.copy()
ev['gap'] = ev.groupby('cookie_id').event_ts.diff().dt.total_seconds()

cand = ev.groupby('cookie_id').agg(
    n_events=('event_ts', 'size'),
    first_ts=('event_ts', 'min'),
    last_ts=('event_ts', 'max'),
    gap_median=('gap', 'median'),
    gap_std=('gap', 'std'),
    gap_max=('gap', 'max'),
    max_page=('search_page', 'max'),
    n_queries=('search_query', 'nunique'),
    n_categories=('item_category', 'nunique'),
    n_locations=('item_location', 'nunique'),
    n_ua=('user_agent', 'nunique'),
).reset_index()
cand['duration_min'] = (cand.last_ts - cand.first_ts).dt.total_seconds() / 60
cand = cand.merge(train[['cookie_id', 'target']], on='cookie_id')

cand.drop(columns=['cookie_id', 'first_ts', 'last_ts']).groupby('target').median().T

target,0,1
n_events,12.0000,20.000000
gap_median,60.5000,25.000000
gap_std,1751.0893,1378.387444
gap_max,5894.0000,6053.000000
max_page,4.0000,7.000000
n_queries,3.0000,4.000000
n_categories,2.0000,2.000000
n_locations,5.0000,8.000000
n_ua,1.0000,1.000000
duration_min,138.7750,151.533333


**Выводы (медианы, 0 — люди, 1 — боты):**
- `gap_median`: 60 с против 25 с — боты кликают быстрее;
- `max_page`: 4 против 7 — боты листают выдачу глубже;
- `n_locations`: 5 против 8 — боты смотрят больше городов;
- `n_events`: 12 против 20 — боты делают больше действий;
- `n_categories`, `n_ua`, `duration_min`, `gap_max` почти не отличаются: боты тоже работают часами и делают длинные перерывы.

In [13]:
ev_t = ev.merge(train[['cookie_id', 'target']], on='cookie_id')
pd.crosstab(ev_t.target, ev_t.event_name, normalize='index').T.round(3)

target,0,1
event_name,,
contact_chat_open,0.020,0.014
contact_message_sent,0.010,0.008
contact_phone_show,0.037,0.026
favorite_add,0.066,0.027
item_view,0.365,0.437
login,0.022,0.008
photo_swipe,0.123,0.070
search_results_view,0.305,0.352
seller_page_view,0.053,0.059


**Выводы:** люди чаще добавляют в избранное (6,6% против 2,7%), логинятся (2,2% против 0,8%) и листают фото (12,3% против 7,0%). Боты чаще просматривают объявления и выдачу.

### 5.1. User-Agent

In [14]:
ua = ev_tr.merge(train[['cookie_id', 'target']], on='cookie_id')

n_ua = ua.groupby('cookie_id').agg(n_ua=('user_agent', 'nunique'), target=('target', 'first'))
print(n_ua.groupby('target').n_ua.agg(['mean', lambda s: (s > 1).mean()]), '\n')

for t in [0, 1]:
    print('target =', t)
    print(ua[ua.target == t].user_agent.value_counts(normalize=True).head(5).round(3), '\n')

            mean  <lambda_0>
target                      
0       1.135793    0.073097
1       1.112347    0.056730 

target = 0
user_agent
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 YaBrowser/23.0.0.0 Safari/537.36    0.012
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 YaBrowser/23.1.0.0 Safari/537.36    0.011
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36                       0.011
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36                       0.011
Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36                 0.011
Name: proportion, dtype: float64 

target = 1
user_agent
Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/118.0.0.0 Safari/

In [15]:
# разбираем строку UA на части
s = ua.user_agent
ua['family'] = np.select(
    [
        s.str.startswith('Avito/'),
        s.str.contains('HeadlessChrome'),
        s.str.contains('YaBrowser'),
        s.str.contains('Firefox'),
        s.str.contains('Chrome'),
    ],
    ['app', 'headless', 'yandex', 'firefox', 'chrome'],
    default='other',
)
# основная версия: у приложения — Avito/122, у браузеров — Chrome/120 или Firefox/121
ua['version'] = s.str.extract(r'(?:Avito|Chrome|Firefox)/(\d+)')[0].astype(float)
# отставание от самой свежей версии этого же семейства
ua['version_lag'] = ua.groupby('family').version.transform('max') - ua.version

# одна строка на куку: берём первый UA
per_cookie = ua.groupby('cookie_id').agg(
    target=('target', 'first'),
    family=('family', 'first'),
    version_lag=('version_lag', 'first'),
)
print(pd.crosstab(per_cookie.family, per_cookie.target, normalize='columns').round(3), '\n')
print(per_cookie.groupby(['family', 'target']).version_lag.median().unstack().round(1))

target        0      1
family                
app       0.191  0.219
chrome    0.518  0.388
firefox   0.128  0.138
headless  0.021  0.083
other     0.033  0.059
yandex    0.109  0.112 

target      0    1
family            
app       2.0  2.0
chrome    4.0  6.0
firefox   6.0  7.0
headless  6.0  6.0
other     NaN  NaN
yandex    5.0  6.0


**Выводы:** у ботов чаще встречаются `HeadlessChrome` (8% против 2%) и редкие браузеры, у людей — обычный Chrome. Отставание версии браузера у ботов чуть больше. Сама строка UA редко меняется внутри куки (у ~6–7% кук больше одного UA).

### 5.2. Курсор (pointer) на web

Координаты курсора есть только у web-событий. Сначала — есть ли курсор вообще.

In [16]:
web = ev_tr[ev_tr.platform == 'web'].merge(train[['cookie_id', 'target']], on='cookie_id')
p = web.groupby('cookie_id').agg(
    target=('target', 'first'),
    share_no_pointer=('pointer_x', lambda s: s.isna().mean()),
    n_unique_x=('pointer_x', 'nunique'),
)
print(p.groupby('target')[['share_no_pointer', 'n_unique_x']].median(), '\n')
print('доля кук, у которых курсора нет ни в одном web-событии:')
print(p.groupby('target').share_no_pointer.agg(all_missing=lambda s: (s == 1).mean()))

        share_no_pointer  n_unique_x
target                              
0               0.066667         6.0
1               1.000000         0.0 

доля кук, у которых курсора нет ни в одном web-событии:
        all_missing
target             
0          0.287859
1          0.573129


У большинства ботов на web координат курсора нет совсем (57% против 29% у людей) — скрипт не двигает мышь. Если же курсор у куки есть, он есть в ~95% её событий и у людей, и у ботов, поэтому «частичная» доля пропусков сигнала не несёт.

Главное отличие — **форма облака точек**. Координаты целые, x от 0 до 1920, y от 0 до 1080. Сравним разброс точек куки с разбросом равномерного распределения по экрану (std = длина стороны / √12).

In [17]:
W, H = 1920, 1080  # размер экрана: максимум координат (все точки train и test лежат в [0, 1920] x [0, 1080])

pts = web[web.pointer_x.notna()]
g = pts.groupby('cookie_id')
cloud = pd.DataFrame({
    'target': g.target.first(),
    'n_points': g.size(),
    'std_x_norm': g.pointer_x.std() / (W / np.sqrt(12)),
    'std_y_norm': g.pointer_y.std() / (H / np.sqrt(12)),
    'std_px': np.sqrt((g.pointer_x.var() + g.pointer_y.var()) / 2),
    'aniso': np.log(g.pointer_x.std() / g.pointer_y.std()),
    'on_edge': ((pts.pointer_x <= 0) | (pts.pointer_x >= W) | (pts.pointer_y <= 0) | (pts.pointer_y >= H)).groupby(pts.cookie_id).mean(),
})
cloud[cloud.n_points >= 5].groupby('target').median().round(3).T

target,0,1
n_points,13.000,19.000
std_x_norm,0.999,0.369
std_y_norm,0.995,0.641
std_px,448.034,207.273
aniso,0.577,0.039
on_edge,0.000,0.035


**Выводы (медианы по кукам с ≥5 точками, 0 — люди, 1 — боты):**
- у людей нормированный разброс ≈ 1 по обеим осям — точки равномерно покрывают экран 1920×1080;
- у ботов разброс ~200 px **одинаковый по обеим осям**: это круглое облако вокруг точки, а не весь экран. Анизотропия `log(std_x/std_y)` у людей ≈ log(1920/1080) = 0.58, у ботов ≈ 0;
- облако ботов обрезается краями экрана, поэтому у них часто встречаются точки ровно на границе (x = 0, y = 0, y = 1080), у людей — почти никогда.

Похоже, скрипт генерирует координаты «клика» из нормального распределения вокруг центра элемента и обрезает их по экрану. Это поведение, а не техническая характеристика, поэтому из него строятся признаки (раздел 6.2).

## 6. Признаки

Признаки считаются сразу для всех кук train и test: для признаков одной куки это ничего не меняет, а признакам по связям между куками (добавляются позже) нужны события всех кук. Каждая функция возвращает таблицу с индексом `cookie_id`, пропуски потом заполняются `-1`.

In [18]:
meta = pd.concat([train.drop(columns='target'), test], ignore_index=True)
ev_all = pd.concat([ev_tr, ev_te]).sort_values(['cookie_id', 'event_ts'], kind='mergesort').reset_index(drop=True)
print(len(meta), 'кук,', len(ev_all), 'событий в окнах')

16000 кук, 283830 событий в окнах


### 6.1. Базовые признаки: объём, ритм, поиск, типы событий, возраст куки

In [19]:
def f_base(ev, meta):
    """Признаки исходной версии решения (этап 1 журнала)."""
    ev = ev.copy()
    ev['gap'] = ev.groupby('cookie_id').event_ts.diff().dt.total_seconds()  # пауза до предыдущего события, сек
    ev['gap_short'] = ev.gap.where(ev.gap < 600)  # пауза, только если меньше 10 минут, иначе NaN
    ev['is_fast'] = ev.gap < 3                    # True, если пауза меньше 3 секунд
    ev['is_break'] = ev.gap > 1800                # True, если пауза больше 30 минут

    g = ev.groupby('cookie_id')
    f = pd.DataFrame({
        # объём и разнообразие
        'n_events': g.size(),
        'item_nunique': g.item_id.nunique(),
        'n_locations': g.item_location.nunique(),
        'n_queries': g.search_query.nunique(),
        # ритм
        'gap_median': g.gap.median(),
        'gap_short_mean': g.gap_short.mean(),
        'gap_short_std': g.gap_short.std(),
        'share_fast': g.is_fast.mean(),
        'n_sessions': g.is_break.sum() + 1,
        # глубина поиска
        'max_page': g.search_page.max(),
    })
    # доли типов событий у куки
    share = pd.crosstab(ev.cookie_id, ev.event_name, normalize='index')
    share.columns = ['share_' + c for c in share.columns]
    f = f.join(share)
    # возраст куки на начало окна, часы
    m = meta.set_index('cookie_id')
    f['cookie_age_h'] = (m.window_start_ts - m.cookie_created_at).dt.total_seconds() / 3600
    return f

### 6.2. Курсор

In [20]:
def f_pointer(ev):
    """Признаки формы облака точек курсора (только web; у android/ios курсора нет -> NaN)."""
    web = ev[ev.platform == 'web']
    pts = web[web.pointer_x.notna()]
    g = pts.groupby('cookie_id')
    f = pd.DataFrame(index=pd.Index(web.cookie_id.unique(), name='cookie_id'))
    f['n_ptr'] = g.size()
    f['n_ptr'] = f.n_ptr.fillna(0)                                   # 0 — web-кука без курсора
    f['share_no_pointer'] = 1 - f.n_ptr / web.groupby('cookie_id').size()
    # дисперсия по каждой оси, нормированная на дисперсию равномерного распределения по экрану (L^2 / 12)
    vx = g.pointer_x.var() / (W ** 2 / 12)
    vy = g.pointer_y.var() / (H ** 2 / 12)
    v = (vx + vy) / 2
    f['ptr_spread'] = np.sqrt(v)                                     # у людей ~1, у ботов ~0.5
    f['ptr_xstd_n'] = np.sqrt(vx)
    f['ptr_ystd_n'] = np.sqrt(vy)
    # насколько уверенно разброс меньше равномерного с учётом числа точек:
    # std выборочной нормированной дисперсии равномерного распределения ≈ 0.894 / sqrt(n) на ось
    f['ptr_spread_z'] = (v - 1) * np.sqrt(2 * f.n_ptr) / 0.894
    f['ptr_std_px'] = np.sqrt((g.pointer_x.var() + g.pointer_y.var()) / 2)
    f['ptr_aniso'] = np.log(g.pointer_x.std() / g.pointer_y.std())   # экран: log(1920/1080) = 0.58, круглое облако: 0
    f['ptr_kurt'] = (g.pointer_x.kurt() + g.pointer_y.kurt()) / 2    # равномерное: -1.2, нормальное: 0
    on_edge = (pts.pointer_x <= 0) | (pts.pointer_x >= W) | (pts.pointer_y <= 0) | (pts.pointer_y >= H)
    f['ptr_clip_share'] = on_edge.groupby(pts.cookie_id).mean()     # облако бота обрезается краями экрана
    f['ptr_clip_n'] = on_edge.groupby(pts.cookie_id).sum()
    # черновой признак первой версии: число различных x (по всем событиям, у не-web = 0)
    f['n_unique_pointer_x'] = g.pointer_x.nunique()
    f = f.reindex(ev.cookie_id.unique())
    f['n_unique_pointer_x'] = f.n_unique_pointer_x.fillna(0)
    return f.replace([np.inf, -np.inf], np.nan)

### 6.3. Ритм внутри сессий

Базовые `gap_short_*` смотрят только на паузы короче 10 минут. Но есть и «медленные» боты с паузами 150–900 с, которых этот порог обрезает. Здесь сессия — цепочка событий с паузами ≤ 30 минут (как в `n_sessions`), и ритм считается по всем паузам внутри сессий.

In [21]:
def f_rhythm(ev):
    """Регулярность и темп пауз внутри сессий."""
    gap = ev.groupby('cookie_id').event_ts.diff().dt.total_seconds()
    new_session = gap.isna() | (gap > 1800)
    session_id = new_session.astype(int).groupby(ev.cookie_id).cumsum()
    gin = gap.where(gap <= 1800)                         # паузы внутри сессий
    g = gin.groupby(ev.cookie_id)
    # коэффициент вариации пауз: у скрипта с планировщиком и джиттером ритм ровнее, чем у человека
    f = pd.DataFrame({'gin_cv': g.std() / g.mean()})
    sec = (ev.event_ts - ev.event_ts.min()).dt.total_seconds()
    s = sec.groupby([ev.cookie_id, session_id])
    sessions = pd.DataFrame({'n': s.size(), 'dur': s.max() - s.min()})
    by_cookie = sessions.groupby(level=0)
    f['sess_n_mean'] = by_cookie.n.mean()                # среднее число событий в сессии
    # темп: событий в секунду внутри сессий (= 1 / средняя пауза внутри сессий)
    f['sess_rate'] = (sessions.n - 1).groupby(level=0).sum() / by_cookie.dur.sum().replace(0, np.nan)
    return f


rhythm = f_rhythm(ev_tr).join(train.set_index('cookie_id').target)
rhythm.groupby('target').median().round(3).T

target,0,1
gin_cv,0.896,0.621
sess_n_mean,5.000,8.000
sess_rate,0.014,0.030


**Выводы:** паузы внутри сессий у ботов ровнее (коэффициент вариации ~0.6 против ~0.9 у людей), темп вдвое выше, сессии длиннее. Точных повторов пауз почти нет (у ботов задержка со случайным джиттером), автокорреляция пауз около нуля у обоих классов, поэтому такие признаки не строились. Признаки времени суток (доля ночных событий и т. п.) тоже проверялись: боты относительно активнее ночью, но на уровне куки сигнал слабый, и rolling-origin падал.

### 6.4. Контент: города, категории, продавцы, согласованность поиска и просмотров

Что именно кука смотрит: сколько категорий и городов, насколько они сосредоточены, совпадают ли открытые объявления с её же поиском, как часто у объявлений не заполнены атрибуты.

In [22]:
BIG4 = ['moskva', 'sankt-peterburg', 'novosibirsk', 'ekaterinburg']
CONTACTS = ['contact_phone_show', 'contact_chat_open', 'contact_message_sent']


def entropy_top(ev, col):
    """Энтропия распределения значений col внутри куки и доля самого частого значения."""
    cnt = ev[['cookie_id', col]].dropna().groupby(['cookie_id', col]).size()
    p = cnt / cnt.groupby(level=0).transform('sum')
    return -(p * np.log(p)).groupby(level=0).sum(), p.groupby(level=0).max()


def share_where(mask, valid, by):
    """Доля mask среди строк, где valid, по куке."""
    return mask.astype(float).where(valid).groupby(by).mean()


def f_content(ev):
    ck, st = ev.cookie_id, ev.seller_type
    g = ev.groupby('cookie_id')
    f = pd.DataFrame(index=pd.Index(ck.unique(), name='cookie_id'))
    f['n_categories'] = g.item_category.nunique()
    f['cat_entropy'], f['cat_top_share'] = entropy_top(ev, 'item_category')
    f['loc_entropy'], f['loc_top_share'] = entropy_top(ev, 'item_location')
    f['big4_share'] = share_where(ev.item_location.isin(BIG4), ev.item_location.notna(), ck)
    is_contact = ev.event_name.isin(CONTACTS)
    f['pro_share'] = share_where(st == 'pro', st.notna(), ck)
    f['pro_share_contact'] = share_where(st == 'pro', st.notna() & is_contact, ck)
    f['n_pro_contacts'] = ((st == 'pro') & is_contact).groupby(ck).sum()
    # доли незаполненных атрибутов объявления
    f['seller_na_share'] = share_where(st.isna(), ev.item_id.notna(), ck)
    f['cat_na_share'] = share_where(ev.item_category.isna(), ev.event_name != 'login', ck)
    f['loc_na_share'] = share_where(ev.item_location.isna(), ev.event_name != 'login', ck)

    s = ev[ev.event_name == 'search_results_view']
    items = ev[ev.item_id.notna()]
    # доминирующая категория поисковых выдач куки и доля товаров из неё
    top = (s.dropna(subset=['item_category']).groupby(['cookie_id', 'item_category']).size().rename('n').reset_index()
           .sort_values(['cookie_id', 'n', 'item_category'], ascending=[True, False, True])
           .drop_duplicates('cookie_id').set_index('cookie_id').item_category)
    m = items.cookie_id.map(top)
    f['item_cat_match_query'] = share_where(items.item_category == m, m.notna() & items.item_category.notna(), items.cookie_id)
    # какая доля категорий / городов открытых товаров встречалась в поисковой выдаче этой же куки
    for col, name in [('item_category', 'item_cat_in_queries'), ('item_location', 'item_loc_in_search')]:
        seen = s[['cookie_id', col]].dropna().drop_duplicates().assign(in_search=1.0)
        it = items[['cookie_id', col]].dropna().drop_duplicates().merge(seen, how='left', on=['cookie_id', col])
        f[name] = it.in_search.fillna(0).groupby(it.cookie_id).mean()
    f['n_search_locs'] = s.groupby('cookie_id').item_location.nunique()
    f['n_query_loc'] = s.dropna(subset=['search_query', 'item_location']).drop_duplicates(
        ['cookie_id', 'search_query', 'item_location']).groupby('cookie_id').size()
    qlp = s.assign(loc=s.item_location.fillna('')).drop_duplicates(['cookie_id', 'search_query', 'loc', 'search_page'])
    f['n_query_loc_page'] = qlp.groupby('cookie_id').size()
    f['n_search'] = s.groupby('cookie_id').size()
    f['query_repeat'] = f.n_search / f.n_query_loc_page                   # повторные просмотры одной и той же выдачи
    f['items_per_loc'] = g.item_id.nunique() / g.item_location.nunique().replace(0, np.nan)
    f['item_repeat'] = items.groupby('cookie_id').size() / items.groupby('cookie_id').item_id.nunique()
    f['items_per_search'] = items.groupby('cookie_id').item_id.nunique() / f.n_search
    return f


content = f_content(ev_tr).join(train.set_index('cookie_id').target)
content.groupby('target')[['cat_entropy', 'cat_top_share', 'loc_entropy', 'n_search_locs', 'item_cat_match_query',
                           'cat_na_share', 'n_search', 'items_per_search']].median().round(3).T

target,0,1
cat_entropy,0.637,0.410
cat_top_share,0.688,0.857
loc_entropy,1.213,1.835
n_search_locs,2.000,4.000
item_cat_match_query,0.611,0.868
cat_na_share,0.030,0.056
n_search,4.000,7.000
items_per_search,1.680,1.455


**Выводы (медианы, 0 — люди, 1 — боты):** бот собирает данные по одной категории, но во многих городах: энтропия категорий ниже (0.41 против 0.64), доля главной категории выше (0.86 против 0.69), а энтропия городов выше (1.84 против 1.21), поисков по разным городам вдвое больше (4 против 2). Открытые ботом объявления чаще из категории его же поиска (0.87 против 0.61) — он систематически обходит выдачу, а человек отвлекается на другое. Поисковых выдач у бота больше (7 против 4), а открытых объявлений на одну выдачу меньше. У объявлений, которые смотрит бот, чаще не заполнены категория и город.

### 6.5. Сборка

In [23]:
def build_features(ev, meta):
    """Одна строка признаков на каждую куку meta (в том же порядке); пропуски -> -1."""
    parts = [f_base(ev, meta), f_pointer(ev), f_rhythm(ev), f_content(ev)]
    f = pd.concat(parts, axis=1)
    # -1 вместо 0: «данных нет» (например, пауза у куки с одним событием) не должно выглядеть как «пауза 0 секунд»
    return meta[['cookie_id']].join(f, on='cookie_id').fillna(-1)


X = build_features(ev_all, meta)
X_tr = X.iloc[:len(train)].reset_index(drop=True)
X_te = X.iloc[len(train):].reset_index(drop=True)
assert (X_tr.cookie_id.values == train.cookie_id.values).all() and (X_te.cookie_id.values == test.cookie_id.values).all()
y_tr = train.target.values

EVENT_SHARES = [c for c in X.columns if c.removeprefix('share_') in set(events.event_name)]
FEATURE_GROUPS = {
    'baseline': ['n_events', 'item_nunique'],
    'rhythm_search': ['gap_median', 'max_page', 'n_locations', 'n_queries'],
    'event_shares': EVENT_SHARES,
    'cookie_age': ['cookie_age_h'],
    'sessions': ['gap_short_mean', 'gap_short_std', 'share_fast', 'n_sessions'],
    'pointer_draft': ['share_no_pointer', 'n_unique_pointer_x'],
    'pointer': ['n_ptr', 'share_no_pointer', 'ptr_spread', 'ptr_xstd_n', 'ptr_ystd_n', 'ptr_spread_z',
                'ptr_std_px', 'ptr_aniso', 'ptr_kurt', 'ptr_clip_share', 'ptr_clip_n'],
    'rhythm': ['gin_cv', 'sess_rate'],
    'rhythm_sess_len': ['sess_n_mean'],
    'content': ['n_categories', 'cat_entropy', 'cat_top_share', 'loc_entropy', 'loc_top_share', 'big4_share',
                'pro_share', 'pro_share_contact', 'n_pro_contacts', 'seller_na_share', 'cat_na_share', 'loc_na_share',
                'item_cat_match_query', 'item_cat_in_queries', 'item_loc_in_search', 'n_search_locs', 'n_query_loc',
                'n_query_loc_page', 'n_search', 'query_repeat', 'items_per_loc', 'item_repeat', 'items_per_search'],
}
assert set(c for cols in FEATURE_GROUPS.values() for c in cols) == set(X.columns) - {'cookie_id'}

print(X.shape[1] - 1, 'признаков')
X_tr.head()

58 признаков


,cookie_id,n_events,item_nunique,n_locations,n_queries,gap_median,gap_short_mean,gap_short_std,share_fast,n_sessions,...,item_cat_in_queries,item_loc_in_search,n_search_locs,n_query_loc,n_query_loc_page,n_search,query_repeat,items_per_loc,item_repeat,items_per_search
0,ck_54a059eb7d3ea68b,7,4,5,3,21.0,27.833333,29.949402,0.142857,1,...,0.333333,0.250000,2.0,3.0,3.0,3.0,1.0,0.800000,1.000000,1.333333
1,ck_7e4de46eeab82974,41,19,12,5,57.5,46.617647,31.594989,0.024390,7,...,1.000000,0.400000,6.0,7.0,8.0,8.0,1.0,1.583333,1.684211,2.375000
2,ck_9320229ef6304522,36,19,9,10,22.0,28.642857,27.068109,0.027778,7,...,0.500000,0.428571,5.0,10.0,13.0,13.0,1.0,2.111111,1.052632,1.461538
3,ck_30ccd25bc1714ed9,21,10,4,3,54.5,49.500000,26.535769,0.000000,3,...,1.000000,0.750000,3.0,3.0,3.0,3.0,1.0,2.500000,1.800000,3.333333
4,ck_a77c5f05948cdeef,28,16,6,6,52.0,65.173913,53.287942,0.000000,4,...,0.750000,0.250000,3.0,5.0,6.0,6.0,1.0,2.666667,1.375000,2.666667


Не используются как признаки: сырые `cookie_id`, `item_id`, даты окна (описывают «кто и когда», а не поведение) и любые события после окна.

## 7. Валидация

Test идёт после train по времени, поэтому исходно валидация была одним временным разрезом: обучение на окнах до 2026-04-17, проверка на трёх последних днях. Проблема — в этих днях всего 160 ботов, и P@R0.7 на них очень шумная: смена seed двигает метрику на ±0.02–0.03, а бутстрэп-интервал ещё шире (см. 7.1). На таком шуме нельзя отличить полезный признак от случайности.

Поэтому каждый вариант оценивается тремя способами:

| Схема | Как устроена | Роль |
|---|---|---|
| **CV по дням** (основная) | 7 фолдов, в каждом 2 дня train целиком; OOF-предсказания всех 11 091 кук, одна P@R0.7 по всем 899 ботам | самый низкий шум |
| **Rolling-origin** | 5 блоков по 2 дня (04-10…04-19), модель учится только на днях **раньше** блока; P@R0.7 по объединённым предсказаниям блоков | проверка, что улучшение сохраняется при обучении только на прошлом — как у настоящего test |
| **Временной holdout** | обучение до 04-17, проверка на 04-17…04-19 | прежняя схема, оставлена для сравнения |

Куки одного дня никогда не разрываются между обучением и проверкой. Метрика везде считается официальной функцией `precision_at_recall`.

**Шум от seed.** LightGBM случайно выбирает строки и признаки для каждого дерева, поэтому один и тот же набор признаков на разных seed даёт P@R0.7 с разбросом ~0.01. Каждый вариант обучается на 10 seed, а два варианта сравниваются **парно** — на одних и тех же seed: считается средняя разница Δ и t-статистика `t = mean(Δ) / (std(Δ) / √10)`.

**Правило принятия изменения:** метрика значимо растёт хотя бы на одной из двух основных схем — CV по дням или rolling-origin (Δ > 0 и t ≥ 2), — и ни на одной из них значимо не падает (нет Δ < 0 при t ≤ −2). Holdout показывается для справки и в решении не участвует.

Почему так:
- Сначала решения принимались только по CV по дням. Но возраст куки по CV дал +0.004 (в пределах шума seed), а по rolling-origin — значимые +0.017: изменение полезно для прогноза «на будущее», и правило «решает только CV» его теряло. Поэтому две основные схемы равноправны.
- Пробовали и вариант, где значимое падение на holdout запрещает изменение. Он отклонял переход RF → LightGBM (CV +0.031, rolling +0.042, holdout −0.043). Но t-статистика по seed учитывает только случайность модели, а не то, что в holdout всего 160 ботов: бутстрэп-интервал holdout шириной ±0.17 (раздел 7.1) на порядок больше таких разниц. Поэтому holdout в решениях не участвует.

In [24]:
from joblib import Parallel, delayed

N_JOBS = -1  # обучения внутри оценки независимы и идут параллельно; у каждой модели 1 поток, поэтому результат не зависит от числа процессов


def make_model(kind='lgbm', seed=SEED):
    """Возвращает модель с фиксированными гиперпараметрами и seed."""
    if kind == 'rf':
        return RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=seed, n_jobs=1)
    return LGBMClassifier(
        n_estimators=500,
        learning_rate=0.03,
        num_leaves=31,
        min_child_samples=20,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        random_state=seed,
        n_jobs=1,
        verbose=-1,
    )


def fit_predict(X_fit, y_fit, X_pred, kind, seed):
    model = make_model(kind, seed)
    model.fit(X_fit, y_fit)
    return model.predict_proba(X_pred)[:, 1]


day = train.window_start_ts.dt.normalize()
days = sorted(day.unique())
cv_fold = day.map({dd: i // 2 for i, dd in enumerate(days)}).values          # 7 фолдов по 2 дня
roll_blocks = [days[i:i + 2] for i in range(4, len(days), 2)]                  # 5 блоков по 2 дня
is_holdout = (train.window_start_ts >= VALID_START).values
SCHEMES = ['cv', 'roll', 'holdout']


def evaluate(X, cols, kind='lgbm', seeds=SEEDS):
    """P@R0.7 набора признаков по трём схемам валидации на каждом seed + средний OOF-прогноз CV."""
    A, y = X[cols], y_tr
    tasks = []  # (seed, схема, какие куки в обучении, какие предсказываем)
    for seed in seeds:
        tasks += [(seed, 'cv', cv_fold != k, cv_fold == k) for k in np.unique(cv_fold)]
        tasks += [(seed, 'roll', (day < b[0]).values, day.isin(b).values) for b in roll_blocks]
        tasks.append((seed, 'holdout', ~is_holdout, is_holdout))
    preds = Parallel(n_jobs=N_JOBS)(
        delayed(fit_predict)(A[fit], y[fit], A[pred], kind, seed) for seed, _, fit, pred in tasks
    )
    scores = {sch: [] for sch in SCHEMES}
    oof = np.zeros(len(y))
    for seed in seeds:
        p = {sch: np.full(len(y), np.nan) for sch in SCHEMES}
        for (s, sch, _, pred), pr in zip(tasks, preds):
            if s == seed:
                p[sch][pred] = pr
        for sch in SCHEMES:
            ok = ~np.isnan(p[sch])
            scores[sch].append(precision_at_recall(y[ok], p[sch][ok]))
        oof += p['cv'] / len(seeds)
    return {'scores': {k: np.array(v) for k, v in scores.items()}, 'oof': oof}


def fmt(r):
    s = r['scores']
    return ' | '.join(f'{k} {s[k].mean():.4f} ± {s[k].std():.4f}' for k in SCHEMES)


def paired(new, ref, sch):
    """Средняя разница метрики на одних и тех же seed и её t-статистика."""
    d = new['scores'][sch] - ref['scores'][sch]
    return d.mean(), d.mean() / (d.std(ddof=1) / np.sqrt(len(d)) + 1e-12)

### 7.1. Насколько шумны схемы валидации

Бутстрэп по кукам: 95%-интервал P@R0.7 на трёх днях holdout и на всех OOF-предсказаниях CV (20 признаков первой версии, LightGBM).

In [25]:
def bootstrap_ci(y, p, n=300, seed=SEED):
    """95%-интервал метрики при случайной перевыборке кук."""
    rng = np.random.default_rng(seed)
    vals = [precision_at_recall(y[idx], p[idx]) for idx in rng.integers(0, len(y), (n, len(y)))]
    return np.percentile(vals, [2.5, 97.5]).round(3)


BASE_COLS = [c for gname in ['baseline', 'rhythm_search', 'event_shares', 'cookie_age', 'sessions'] for c in FEATURE_GROUPS[gname]]
r_base = evaluate(X_tr, BASE_COLS)
print(fmt(r_base))
print('ботов: holdout', y_tr[is_holdout].sum(), '| CV', y_tr.sum())
print('95% CI, holdout (3 дня):', bootstrap_ci(y_tr[is_holdout], r_base['oof'][is_holdout]))
print('95% CI, CV (все дни):   ', bootstrap_ci(y_tr, r_base['oof']))

cv 0.4509 ± 0.0145 | roll 0.4402 ± 0.0142 | holdout 0.4022 ± 0.0200
ботов: holdout 160 | CV 899
95% CI, holdout (3 дня): [0.259 0.599]
95% CI, CV (все дни):    [0.403 0.544]


Интервал на трёх днях holdout заметно шире, чем на всех OOF-предсказаниях, поэтому решения принимаются по CV по дням и rolling-origin, а holdout показывается для справки.

## 8. Журнал экспериментов

Каждая строка — изменение относительно последнего **принятого** шага: добавить группы признаков (`add`), убрать (`remove`) или сменить модель (`model`). Строка сравнивается с последним принятым шагом парно по 10 seed, решение выносится по правилу из раздела 7. Все строки пересчитываются здесь же, поэтому журнал воспроизводится.

In [26]:
EXPERIMENTS = [
    # этап 1: исходные шаги, пересчитанные на новой валидации
    dict(name='Baseline quickstart', add=['baseline'], model='rf'),
    dict(name='+ ритм и глубина поиска', add=['rhythm_search']),
    dict(name='+ доли типов событий', add=['event_shares']),
    dict(name='+ возраст куки', add=['cookie_age']),
    dict(name='+ ритм внутри сессий', add=['sessions']),
    dict(name='RF → LightGBM', model='lgbm'),
    # этап 2
    dict(name='+ курсор: черновик (share_no_pointer, n_unique_pointer_x)', add=['pointer_draft']),
    dict(name='+ курсор: форма облака точек (11)', add=['pointer']),
    dict(name='+ ритм: вариация пауз и темп внутри сессий', add=['rhythm']),
    dict(name='+ ритм: средняя длина сессии', add=['rhythm_sess_len']),
    dict(name='+ контент: города, категории, продавцы, согласованность с поиском', add=['content']),
]


def run_journal(experiments):
    groups, kind, ref = [], None, None
    rows, results = [], {}
    for e in experiments:
        g = [x for x in groups if x not in e.get('remove', [])] + e.get('add', [])
        k = e.get('model', kind)
        cols = list(dict.fromkeys(c for x in g for c in FEATURE_GROUPS[x]))
        r = evaluate(X_tr, cols, k)
        row = {'шаг': e['name'], 'модель': k, 'признаков': len(cols)}
        row.update({f'{sch}': round(r['scores'][sch].mean(), 4) for sch in SCHEMES})
        row.update({f'{sch}_std': round(r['scores'][sch].std(), 4) for sch in SCHEMES})
        if ref is None:
            accepted = True
        else:
            diff = {sch: paired(r, ref, sch) for sch in SCHEMES}
            for sch, (d, t) in diff.items():
                row[f'Δ{sch}'], row[f't_{sch}'] = round(d, 4), round(t, 1)
            gain = any(diff[sch][0] > 0 and diff[sch][1] >= 2 for sch in ['cv', 'roll'])
            loss = any(diff[sch][0] < 0 and diff[sch][1] <= -2 for sch in ['cv', 'roll'])
            accepted = gain and not loss
        row['решение'] = 'берём' if accepted else 'не берём'
        if accepted:
            groups, kind, ref = g, k, r
        rows.append(row)
        results[e['name']] = r
        print(f"{e['name']:<58} {fmt(r)} | {row['решение']}", flush=True)
    return pd.DataFrame(rows), groups, kind, results


journal, FINAL_GROUPS, FINAL_KIND, results = run_journal(EXPERIMENTS)
FINAL_COLS = list(dict.fromkeys(c for x in FINAL_GROUPS for c in FEATURE_GROUPS[x]))
journal

Baseline quickstart                                        cv 0.0974 ± 0.0004 | roll 0.0961 ± 0.0005 | holdout 0.0938 ± 0.0017 | берём
+ ритм и глубина поиска                                    cv 0.2868 ± 0.0039 | roll 0.2635 ± 0.0036 | holdout 0.2684 ± 0.0108 | берём
+ доли типов событий                                       cv 0.3358 ± 0.0092 | roll 0.3175 ± 0.0082 | holdout 0.3289 ± 0.0102 | берём
+ возраст куки                                             cv 0.3398 ± 0.0059 | roll 0.3343 ± 0.0083 | holdout 0.3478 ± 0.0126 | берём
+ ритм внутри сессий                                       cv 0.4201 ± 0.0115 | roll 0.3984 ± 0.0116 | holdout 0.4452 ± 0.0057 | берём
RF → LightGBM                                              cv 0.4509 ± 0.0145 | roll 0.4402 ± 0.0142 | holdout 0.4022 ± 0.0200 | берём
+ курсор: черновик (share_no_pointer, n_unique_pointer_x)  cv 0.4625 ± 0.0077 | roll 0.4285 ± 0.0130 | holdout 0.4291 ± 0.0155 | не берём
+ курсор: форма облака точек (11)                   

,шаг,модель,признаков,cv,roll,holdout,cv_std,roll_std,holdout_std,решение,Δcv,t_cv,Δroll,t_roll,Δholdout,t_holdout
0,Baseline quickstart,rf,2,0.0974,0.0961,0.0938,0.0004,0.0005,0.0017,берём,NaN,NaN,NaN,NaN,NaN,NaN
1,+ ритм и глубина поиска,rf,6,0.2868,0.2635,0.2684,0.0039,0.0036,0.0108,берём,0.1895,141.5,0.1674,130.1,0.1746,48.5
2,+ доли типов событий,rf,15,0.3358,0.3175,0.3289,0.0092,0.0082,0.0102,берём,0.0489,13.8,0.0540,21.0,0.0605,11.8
3,+ возраст куки,rf,16,0.3398,0.3343,0.3478,0.0059,0.0083,0.0126,берём,0.0040,1.2,0.0167,4.9,0.0189,2.7
4,+ ритм внутри сессий,rf,20,0.4201,0.3984,0.4452,0.0115,0.0116,0.0057,берём,0.0803,16.5,0.0642,11.6,0.0973,23.5
5,RF → LightGBM,lgbm,20,0.4509,0.4402,0.4022,0.0145,0.0142,0.0200,берём,0.0309,4.6,0.0417,6.1,-0.0430,-6.0
6,"+ курсор: черновик (share_no_pointer, n_unique...",lgbm,22,0.4625,0.4285,0.4291,0.0077,0.0130,0.0155,не берём,0.0116,1.9,-0.0117,-1.7,0.0269,2.9
7,+ курсор: форма облака точек (11),lgbm,31,0.7435,0.7075,0.6853,0.0085,0.0084,0.0208,берём,0.2926,62.3,0.2673,66.6,0.2832,49.9
8,+ ритм: вариация пауз и темп внутри сессий,lgbm,33,0.7526,0.7239,0.7372,0.0104,0.0120,0.0155,берём,0.0091,1.7,0.0164,3.2,0.0518,6.3
9,+ ритм: средняя длина сессии,lgbm,34,0.7528,0.7157,0.7203,0.0052,0.0122,0.0195,не берём,0.0002,0.0,-0.0082,-1.3,-0.0169,-1.9


**Выводы:**
- Все группы признаков первой версии подтверждаются и на новой валидации. Возраст куки по CV даёт +0.004 (в пределах шума), но значимо улучшает rolling-origin (+0.017, t = 4.9) — принят по правилу из раздела 7.
- LightGBM на тех же признаках лучше RandomForest: CV +0.031 (t = 4.6), rolling-origin +0.042 (t = 6.1). На holdout он хуже на 0.043, но это схема с самым большим выборочным шумом (7.1).
- Черновые признаки курсора (доля событий без курсора и число разных x): CV +0.012 (t = 1.9), rolling-origin −0.012 (t = −1.7) — значимого улучшения нет, в таком виде не берём.
- Признаки формы облака точек курсора — самый большой прирост: CV 0.45 → 0.74, rolling-origin 0.44 → 0.71. Они различают «равномерные» точки людей и круглое облако ботов, обрезанное краями экрана.
- Ритм внутри сессий (вариация пауз и темп): CV +0.009 (t = 1.7), rolling-origin +0.016 (t = 3.2), holdout +0.052 — берём. Средняя длина сессии поверх них ничего не добавляет.
- Контент (города, категории, продавцы, согласованность просмотров с поиском): CV 0.75 → 0.79 (t = 8.3), rolling-origin 0.72 → 0.77 (t = 8.4).

## 9. Финальная модель и submission.csv

Модель — последний принятый шаг журнала. Она обучается на всём train; чтобы снизить зависимость от seed, итоговый score — среднее предсказаний моделей с 10 seed.

In [27]:
print('модель:', FINAL_KIND, '| признаков:', len(FINAL_COLS), '| группы:', FINAL_GROUPS)
score = np.mean(Parallel(n_jobs=N_JOBS)(
    delayed(fit_predict)(X_tr[FINAL_COLS], y_tr, X_te[FINAL_COLS], FINAL_KIND, seed) for seed in SEEDS
), axis=0)

sub = pd.DataFrame({'cookie_id': X_te.cookie_id, 'score': score})

# проверки формата: каждая кука из test ровно один раз, score в [0, 1]
assert len(sub) == len(test)
assert sub.cookie_id.is_unique
assert set(sub.cookie_id) == set(test.cookie_id)
assert sub.score.between(0, 1).all()

sub.to_csv('submission.csv', index=False)
sub.head()

модель: lgbm | признаков: 56 | группы: ['baseline', 'rhythm_search', 'event_shares', 'cookie_age', 'sessions', 'pointer', 'rhythm', 'content']


,cookie_id,score
0,ck_315fb710a0e371e7,0.000220
1,ck_a76ee3b3e3e522fd,0.073919
2,ck_94c9a4d382689e82,0.043573
3,ck_8eaf9509ad9462a0,0.000105
4,ck_9a88a5a989cb5bc6,0.002840
